In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from pydantic import BaseModel
import json
import os

load_dotenv()



True

In [8]:
model = ChatNVIDIA(
    model="meta/llama-4-maverick-17b-128e-instruct",
    api_key=os.getenv("KIMI_API"),
    temperature=1,
    top_p=1,
    max_completion_tokens=16384,
)


In [9]:
# ── CELL 3 — Define Structure ─────────────────────────────────────────────────
class Person(BaseModel):
    name: str
    age: int
    job: str
    city: str
    

In [10]:
# ── CELL 4 — TEST 1: Normal Output ───────────────────────────────────────────
print("=" * 50)
print("TEST 1: NORMAL OUTPUT")
print("=" * 50)

normal_response = model.invoke("Tell me about a person named John who is 25 years old, works as an engineer, and lives in Delhi.")
print(normal_response.content)

TEST 1: NORMAL OUTPUT
John is a 25-year-old engineer residing in Delhi. As a young professional, he is likely to be actively engaged in his career and navigating the challenges of adulthood. Here's a more detailed overview of John:

**Personal Life:** John is in the early stages of his career, having likely completed his engineering degree a few years ago. He is at an age where he is building his professional network, gaining experience, and possibly exploring different areas of specialization within his field. Living in Delhi, he is surrounded by a vibrant and fast-paced urban environment, with access to a wide range of cultural, educational, and professional opportunities.

**Professional Life:** As an engineer, John is likely working on designing, developing, or maintaining infrastructure, products, or systems. His specific role could vary widely depending on his branch of engineering (e.g., mechanical, electrical, civil, software). Delhi, being a major hub for various industries, i

In [11]:
# ── CELL 5 — TEST 2: Structured Output ───────────────────────────────────────
print("\n" + "=" * 50)
print("TEST 2: STRUCTURED OUTPUT")
print("=" * 50)

try:
    structured_model = model.with_structured_output(Person)
    structured_response = structured_model.invoke("Give me details about a person named John who is 25 years old, works as an engineer, and lives in Delhi.")
    
    print("✅ Structured output WORKS!")
    print(f"Name : {structured_response.name}")
    print(f"Age  : {structured_response.age}")
    print(f"Job  : {structured_response.job}")
    print(f"City : {structured_response.city}")

except Exception as e:
    print(f"❌ Structured output NOT supported. Error:\n{e}")

    print("\n" + "=" * 50)
    print("FALLBACK: MANUAL JSON OUTPUT")
    print("=" * 50)

    json_response = model.invoke(
        """Return ONLY a valid JSON object, no extra text, no markdown, no explanation.
        Format: {"name": "...", "age": ..., "job": "...", "city": "..."}
        
        Person: John, 25 years old, engineer, lives in Delhi."""
    )

    try:
        data = json.loads(json_response.content)
        print("✅ JSON fallback WORKS!")
        print(f"Name : {data['name']}")
        print(f"Age  : {data['age']}")
        print(f"Job  : {data['job']}")
        print(f"City : {data['city']}")
    except json.JSONDecodeError:
        print("❌ JSON fallback also failed. Raw response:")
        print(json_response.content)


TEST 2: STRUCTURED OUTPUT


c:\Users\PRIYANSHU\Desktop\LANGGRAPGH\myenv\Lib\site-packages\langchain_nvidia_ai_endpoints\chat_models.py:1271: UserWarning: Model 'meta/llama-4-maverick-17b-128e-instruct' is not known to support structured output. Your output may fail at inference time.
  warnings.warn(


✅ Structured output WORKS!
Name : John
Age  : 25
Job  : engineer
City : Delhi
